# How ML Models Work in Production

Let's follow the same penguin model from the first project. We'll fit it locally so this notebook runs independently. Then we'll save the fitted pipeline and use it for prediction. The remaining production decisions are explained in the reference sheet and application design challenge.

## 1. Get the supporting files

If you've downloaded this folder, the files are already present. In Colab, this cell gets the public helper and dataset. It checks the dataset against the version used in the video.

In [1]:
from pathlib import Path
import hashlib
import urllib.request

BASE = "https://raw.githubusercontent.com/G0rav/machine_learning_explained_visually_free/main/09-ml-production/"
for relative in ("production.py", "data/penguins.csv", "data/SOURCE.md"):
    destination = Path(relative)
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(BASE + relative, destination)

expected_hash = "f204db2c753b0937caac3cb35258562c14f073e4bbc76be24b4c51ce22767a93"
assert hashlib.sha256(Path("data/penguins.csv").read_bytes()).hexdigest() == expected_hash
print("Supporting files ready")

Supporting files ready


## 2. Reproduce training once

The helper removes rows with missing required measurements and reproduces the original stratified split. The scaler and logistic regression classifier learn from the training rows only. We leave the test rows unused here because this lesson is about inference, not another model comparison.

In [2]:
import json
import joblib
import numpy as np
import sklearn
from production import FEATURES, fit_training_pipeline, prepare_input, predict_request, predict_batch

pipeline, counts = fit_training_pipeline()
print(json.dumps(counts, indent=2))
print("Pipeline operations:", [name for name, step in pipeline.steps])
print("Training means:", pipeline.named_steps["standardscaler"].mean_)
print("Python dependencies:", "numpy", np.__version__, "scikit-learn", sklearn.__version__, "joblib", joblib.__version__)

{
  "raw_rows": 344,
  "usable_rows": 342,
  "training_rows": 273,
  "sealed_test_rows": 69
}
Pipeline operations: ['standardscaler', 'logisticregression']
Training means: [43.81501832 17.12051282]
Python dependencies: numpy 2.5.3 scikit-learn 1.9.1 joblib 1.6.0


## 3. Preserve the fitted pipeline

Saving the pipeline preserves the scaler's learned statistics and the classifier's learned parameters together. We're saving the fitted object, not just the Python code that constructs a new model. The file below is created by this notebook.

In [3]:
artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
artifact = artifact_dir / "penguins_pipeline.joblib"
joblib.dump(pipeline, artifact)
print("Saved:", artifact)
environment = {"numpy": np.__version__, "scikit-learn": sklearn.__version__, "joblib": joblib.__version__}
(artifact_dir / "environment.json").write_text(json.dumps(environment, indent=2))

Saved: artifacts/penguins_pipeline.joblib


70

## 4. Load before prediction

A serving process would load its artifact before handling requests and keep the fitted object available. Here we reload the file we just created in the same Python environment. Loading does not train a replacement model.

In [4]:
serving_pipeline = joblib.load(artifact)
print([name for name, step in serving_pipeline.steps])
np.testing.assert_allclose(serving_pipeline.named_steps["standardscaler"].mean_, pipeline.named_steps["standardscaler"].mean_)
np.testing.assert_allclose(serving_pipeline.named_steps["logisticregression"].coef_, pipeline.named_steps["logisticregression"].coef_)
print("Fitted statistics and coefficients preserved")

['standardscaler', 'logisticregression']
Fitted statistics and coefficients preserved


## 5. Define the named input

The application collects two measurements in millimeters. JSON preserves the field names and values. The model still needs a numerical row in its expected feature order.

In [5]:
request = {"bill_length_mm": 49.8, "bill_depth_mm": 16.8}
print(json.dumps(request, indent=2))
print("Feature order:", FEATURES)
row = prepare_input(request)
print("Expected:", row)
print("Reversed:", row[:, ::-1])
assert row.tolist() == [[49.8, 16.8]]

{
  "bill_length_mm": 49.8,
  "bill_depth_mm": 16.8
}
Feature order: ('bill_length_mm', 'bill_depth_mm')
Expected: [[49.8 16.8]]
Reversed: [[16.8 49.8]]


## 6. Check inputs before prediction

An absent measurement produces an input error. A string or boolean is not a valid measurement in this contract. Infinity and NaN also fail the check. These checks establish that the input can be processed; they do not establish that the measurement is correct.

In [6]:
invalid_requests = {
    "missing depth": {"bill_length_mm": 49.8},
    "text": {"bill_length_mm": "49.8", "bill_depth_mm": 16.8},
    "boolean": {"bill_length_mm": True, "bill_depth_mm": 16.8},
    "not finite": {"bill_length_mm": float("nan"), "bill_depth_mm": 16.8},
    "infinity": {"bill_length_mm": float("inf"), "bill_depth_mm": 16.8},
    "nonpositive": {"bill_length_mm": 0, "bill_depth_mm": 16.8},
}
for name, candidate in invalid_requests.items():
    try:
        prepare_input(candidate)
    except ValueError as error:
        print(name + ":", error)
    else:
        raise AssertionError("Invalid input was accepted: " + name)

missing depth: Missing fields: bill_depth_mm
text: bill_length_mm must be numeric; booleans and strings are not measurements
boolean: bill_length_mm must be numeric; booleans and strings are not measurements
not finite: bill_length_mm must be a finite positive millimeter measurement
infinity: bill_length_mm must be a finite positive millimeter measurement
nonpositive: bill_length_mm must be a finite positive millimeter measurement


## 7. Keep units explicit

The numeric value `4.98` does not identify its unit. If the interface accepts centimeters, convert to the model's millimeters explicitly. The validator cannot infer that conversion from the number alone.

In [7]:
length_cm = 4.98
length_mm = length_cm * 10
converted_request = {"bill_length_mm": length_mm, "bill_depth_mm": 16.8}
print(f"{length_cm} cm * 10 = {length_mm} mm")
np.testing.assert_allclose(prepare_input(converted_request), row)

4.98 cm * 10 = 49.800000000000004 mm


## 8. Predict with the loaded object

Now the loaded pipeline transforms the ordered input using the training statistics and computes the predicted species. The application can display the result from this response. The model version tells us which reviewed artifact produced it.

In [8]:
response = predict_request(serving_pipeline, request)
print(json.dumps(response, indent=2))
assert response["predicted_species"] == "Gentoo"

{
  "predicted_species": "Gentoo",
  "model_version": "penguins-v1"
}


## 9. Repeated requests do not retrain the model

A second request uses the same learned values. We can check the scaler statistics and classifier coefficients before and after inference. An unchanged object still needs separate monitoring for prediction quality.

In [9]:
before_means = serving_pipeline.named_steps["standardscaler"].mean_.copy()
before_coefficients = serving_pipeline.named_steps["logisticregression"].coef_.copy()
for _ in range(3):
    print(predict_request(serving_pipeline, request))
np.testing.assert_array_equal(before_means, serving_pipeline.named_steps["standardscaler"].mean_)
np.testing.assert_array_equal(before_coefficients, serving_pipeline.named_steps["logisticregression"].coef_)
print("Inference left the learned values unchanged")

{'predicted_species': 'Gentoo', 'model_version': 'penguins-v1'}
{'predicted_species': 'Gentoo', 'model_version': 'penguins-v1'}
{'predicted_species': 'Gentoo', 'model_version': 'penguins-v1'}
Inference left the learned values unchanged


## 10. Process a collection

Batch inference prepares a collection and calls the fitted pipeline for those rows together. On-demand versus batch describes when results are computed. Device versus remote describes where computation happens. The application design exercise asks you to choose both separately.

In [10]:
requests = [request, {"bill_length_mm": 39.1, "bill_depth_mm": 18.7}]
batch_predictions = predict_batch(serving_pipeline, requests)
single_predictions = [predict_request(serving_pipeline, item)["predicted_species"] for item in requests]
print("Batch predictions:", batch_predictions)
assert batch_predictions == single_predictions
assert predict_batch(serving_pipeline, []) == []
print("Single-request and batch results agree")

Batch predictions: ['Gentoo', 'Adelie']
Single-request and batch results agree


## What remains after a prediction works?

A successful response establishes that the operation completed. To assess correctness, compare the prediction with a reliable species label. Track service errors and latency separately from prediction quality. If input patterns change, investigate the measurements and population rather than assuming accuracy has fallen. Train and evaluate a candidate separately before releasing a new model version. Keep the previous artifact available for rollback.

Next, complete [input practice](practice.ipynb), the [application design](project.md), and the [quiz](quiz.md). The [reference sheet](cheat_sheet.md) explains the technical terms.